In [1]:
# pip install ollama pandas
import ollama
import pandas as pd

In [2]:
def build_prompt(ingredients, staples=None, cuisine="No preference", meal_type="Any",
                  diet="None", max_time=30, servings=2, spice_level="Medium",
                  grounding_recipes=None):
    """Construct a structured, constraint-rich prompt for the LLM."""
    staples = staples or []
    all_ingredients = ingredients + staples
    ingredient_list = ", ".join(all_ingredients) if all_ingredients else "whatever is commonly on hand"

    constraints = [
        f"- Servings: {servings}",
        f"- Maximum total cooking time: {max_time} minutes",
        f"- Spice level: {spice_level}",
    ]
    if cuisine != "No preference":
        constraints.append(f"- Cuisine style: {cuisine}")
    if meal_type != "Any":
        constraints.append(f"- Meal type: {meal_type}")
    if diet != "None":
        constraints.append(f"- Dietary restriction: strictly {diet}")

    grounding_block = ""
    if grounding_recipes:
        formatted = "\n".join(
            f"  {i+1}. \"{r['name']}\" \u2014 key ingredients: {r['ingredients']}"
            for i, r in enumerate(grounding_recipes)
        )
        grounding_block = f"""
For inspiration, here are similar real recipes from a recipe database. Use them as a
reference for realistic technique and flavor pairing, but do not just copy one —
synthesize something that best fits the ingredients and constraints below:
{formatted}
"""

    prompt = f"""You are PantryChef, an expert home-cooking assistant. A user wants a recipe
using primarily the ingredients they already have.

Available ingredients: {ingredient_list}
{grounding_block}
Constraints:
{chr(10).join(constraints)}

Instructions:
1. Suggest ONE recipe that uses as many of the available ingredients as possible.
   You may assume basic staples (salt, pepper, oil, water) are available even if not listed.
2. If a key ingredient is missing for a good dish, suggest ONE reasonable substitution.
3. Respond in this exact format:

## [Recipe Name]
**Why this works:** [1-2 sentence rationale tying it to the listed ingredients]

**Ingredients:**
- [quantity] [ingredient]
- ...

**Steps:**
1. [step]
2. ...

**Estimated time:** [X minutes]
**Chef's tip:** [one practical tip]
"""
    return prompt

In [3]:
def load_recipe_dataset(csv_path):
    """Load the Food.com (or similar) recipes CSV."""
    return pd.read_csv(csv_path)


def find_similar_recipes(ingredients, df, top_n=3):
    """Very lightweight retrieval: score recipes by ingredient-token overlap."""
    if df is None or not ingredients:
        return []

    ing_col = next((c for c in df.columns if c.lower() in ("ingredients", "ingredients_raw", "ingredient_list")), None)
    name_col = next((c for c in df.columns if c.lower() in ("name", "title", "recipe_name")), None)
    if ing_col is None or name_col is None:
        print("Couldn't find recognizable name/ingredients columns in the dataset.")
        return []

    tokens = set(t.strip().lower() for t in ingredients if t.strip())

    def score(row_ingredients):
        try:
            text = str(row_ingredients).lower()
        except Exception:
            return 0
        return sum(1 for t in tokens if t in text)

    scored = df.copy()
    scored["_score"] = scored[ing_col].apply(score)
    scored = scored[scored["_score"] > 0].sort_values("_score", ascending=False).head(top_n)

    results = []
    for _, row in scored.iterrows():
        results.append({"name": str(row[name_col]), "ingredients": str(row[ing_col])[:200]})
    return results


# Example (uncomment and point to your downloaded Food.com CSV to enable grounding):
# recipe_df = load_recipe_dataset("RAW_recipes.csv")
recipe_df = None

In [6]:
def ask_pantrychef(ingredients, staples=None, cuisine="No preference", meal_type="Any",
                    diet="None", max_time=30, servings=2, spice_level="Medium",
                    model="llama3.1", temperature=0.8, use_dataset=False):
    """Build the prompt, optionally ground it in the dataset, and ask Ollama for a recipe."""
    grounding_recipes = None
    if use_dataset and recipe_df is not None:
        grounding_recipes = find_similar_recipes(ingredients, recipe_df)

    prompt = build_prompt(
        ingredients, staples=staples, cuisine=cuisine, meal_type=meal_type,
        diet=diet, max_time=max_time, servings=servings, spice_level=spice_level,
        grounding_recipes=grounding_recipes,
    )

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature},
    )

    return response['message']['content'], prompt, grounding_recipes

In [7]:
raw = input("What ingredients do you have? (comma-separated): ")
ingredients = [i.strip() for i in raw.split(",") if i.strip()]

recipe_text, prompt_used, grounding = ask_pantrychef(
    ingredients,
    staples=["salt", "pepper", "olive oil"],
    model="llama3.1",
    use_dataset=False,
)

print("\n🍳 PantryChef says:\n")
print(recipe_text)

What ingredients do you have? (comma-separated):  onion,tomato



🍳 PantryChef says:

## Pantry Pasta Scramble
**Why this works:** This recipe leverages the available ingredients to create a simple, savory dish that can be customized with the few ingredients on hand.

**Ingredients:**
- 1 medium onion
- 2 medium tomatoes
- 2 tablespoons olive oil
- Salt and pepper (to taste)

**Steps:**
1. Chop the onion into small pieces and sauté in 1 tablespoon of olive oil until translucent.
2. Add diced tomatoes to the pan, stirring occasionally for about 5 minutes or until they start breaking down.
3. Use a spatula to create two wells in the mixture; crack an egg (or substitute with 1/4 cup of plain yogurt) into each well and scramble until cooked through.
4. Season with salt and pepper to taste.

**Estimated time:** 20 minutes
**Chef's tip:** To enhance flavor, let the tomato-onion mixture simmer for a minute or two after adding the tomatoes.
